In [ ]:
# ── CPU Allocation ───────────────────────────────────────────────────────────
CPU_MIN = 1
CPU_MAX = 8   # set to os.cpu_count() to use all available cores

import os
_n = max(CPU_MIN, min(CPU_MAX, os.cpu_count() or 1))
for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
             'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_var] = str(_n)
print(f'Allocated {_n} / {os.cpu_count()} CPUs')
# ─────────────────────────────────────────────────────────────────────────────

# Load Streaming Covariance Scaling Data

This local CPU notebook discovers the saved `streaming_covariance_observables` campaign, loads the completed run bundles and the derived scaling-fit outputs, and writes characterization videos/figures from saved data only.

## Notebook Scope

The notebook targets the completed `N20x30`, `N20x40`, and `N20x50` streaming covariance runs for both `perfect_correction` and `postselect`. It uses saved `local_charge_cell.npz` files plus `streaming_covariance_scaling_fits.npz`; it does not run dynamics or require a GPU.

In [ ]:
from __future__ import annotations

import json
import math
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib import colors as mcolors
from matplotlib.animation import FFMpegWriter, FuncAnimation
from IPython.display import Image, Markdown, Video, display

try:
    import imageio_ffmpeg
except ImportError:
    imageio_ffmpeg = None

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR
while ROOT != ROOT.parent and not (ROOT / 'colab_charge_fluctuations').exists():
    ROOT = ROOT.parent
if not (ROOT / 'colab_charge_fluctuations').exists():
    raise FileNotFoundError('Could not locate repo root with colab_charge_fluctuations/')

BUNDLE_ROOT = ROOT / 'colab_charge_fluctuations'
GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
SCALING_OUTPUT_ROOT = BUNDLE_ROOT / 'analysis_outputs' / 'streaming_covariance_scaling_fits'
CHARACTERIZATION_NAME = 'streaming_covariance_characterization'
CHARACTERIZATION_ROOT = BUNDLE_ROOT / 'analysis_outputs' / CHARACTERIZATION_NAME

DATASET_NAME = 'streaming_covariance_observables'
CAMPAIGNS_ROOT = GPU_DATA_ROOT / DATASET_NAME / 'campaigns'
campaign_ids = sorted(p.name for p in CAMPAIGNS_ROOT.iterdir() if p.is_dir())
if len(campaign_ids) != 1:
    raise ValueError(f'Expected one streaming covariance campaign, found {campaign_ids}')
CAMPAIGN_ID = campaign_ids[0]

analysis_root = CHARACTERIZATION_ROOT / CAMPAIGN_ID
figure_root = analysis_root / 'figures'
table_root = analysis_root / 'tables'
figure_root.mkdir(parents=True, exist_ok=True)
table_root.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)

print(f'[repo root] {ROOT}')
print(f'[bundle root] {BUNDLE_ROOT}')
print(f'[campaign id] {CAMPAIGN_ID}')
print(f'[analysis root] {analysis_root}')

## 1. Load Saved Runs

The loader keeps only run folders with both `local_charge_cell.npz` and `streaming_covariance_scaling_fits.npz`. The incomplete `N20x60` run is recorded as skipped.

In [ ]:
ENTROPY_TARGET = 1.0 / 3.0
CORRELATOR_TARGET = 2.0
ROLLING_WINDOW = 5
ROLLING_MIN_PERIODS = 1
ROLLING_CENTER = True
SCATTER_MIN_CYCLE = 20
TARGET_NY = (30, 40, 50)

SOURCE_RUNS_ROOT = GPU_DATA_ROOT / DATASET_NAME / 'campaigns' / CAMPAIGN_ID / 'runs'
FIT_RUNS_ROOT = SCALING_OUTPUT_ROOT / CAMPAIGN_ID / 'runs'


def _json_load(path):
    with Path(path).open('r', encoding='utf-8') as fh:
        return json.load(fh)


def _load_npz(path):
    with np.load(path, allow_pickle=False) as z:
        return {key: z[key] for key in z.files}


def _scalar(value):
    arr = np.asarray(value)
    return arr.item() if arr.shape == () else value


runs = {}
skipped_runs = []
for run_dir in sorted(p for p in SOURCE_RUNS_ROOT.iterdir() if p.is_dir()):
    local_charge_path = run_dir / 'local_charge_cell.npz'
    fit_path = FIT_RUNS_ROOT / run_dir.name / 'streaming_covariance_scaling_fits.npz'
    summary_path = run_dir / 'run_summary.json'
    metrics_path = run_dir / 'scalar_metrics.csv'
    missing = [str(p.name) for p in (local_charge_path, fit_path) if not p.exists()]
    if missing:
        skipped_runs.append({'run_dir': str(run_dir), 'missing': missing})
        continue

    local_payload = _load_npz(local_charge_path)
    fit_payload = _load_npz(fit_path)
    summary = _json_load(summary_path) if summary_path.exists() else {}
    metrics_df = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()

    local_charge = local_payload['local_charge_cell'].astype(np.float64, copy=False)
    sample_indices = local_payload['sample_indices'].astype(np.int64, copy=False)
    cycle_labels = local_payload['cycle_labels'].astype(np.int64, copy=False)
    samples, cycles, nx, ny = local_charge.shape
    protocol = str(_scalar(fit_payload['protocol']))
    geometry_key = str(_scalar(fit_payload['geometry_key']))
    config_id = str(_scalar(fit_payload['config_id']))

    q_raw = 2.0 * (local_charge.sum(axis=(2, 3)) - float(nx * ny))
    q_pct = 100.0 * q_raw / (2.0 * float(nx * ny))
    corr_exponent = np.abs(fit_payload['correlator_loglog_slope'].astype(np.float64, copy=False))

    if q_pct.shape != fit_payload['entropy_slope'].shape:
        raise ValueError(f'{run_dir.name}: charge and fit arrays have inconsistent shapes')

    runs[config_id] = {
        'config_id': config_id,
        'geometry_key': geometry_key,
        'protocol': protocol,
        'Nx': int(nx),
        'Ny': int(ny),
        'samples_actual': int(samples),
        'cycles': int(cycles),
        'sample_indices': sample_indices,
        'cycle_labels': cycle_labels,
        'local_charge_cell': local_charge,
        'q_raw': q_raw,
        'q_pct': q_pct,
        'entropy_slope': fit_payload['entropy_slope'].astype(np.float64, copy=False),
        'entropy_r2': fit_payload['entropy_r2'].astype(np.float64, copy=False),
        'entropy_slope_stderr': fit_payload['entropy_slope_stderr'].astype(np.float64, copy=False),
        'correlator_slope': fit_payload['correlator_loglog_slope'].astype(np.float64, copy=False),
        'correlator_exponent': corr_exponent,
        'correlator_r2': fit_payload['correlator_loglog_r2'].astype(np.float64, copy=False),
        'correlator_slope_stderr': fit_payload['correlator_loglog_slope_stderr'].astype(np.float64, copy=False),
        'summary': summary,
        'metrics_df': metrics_df,
        'paths': {
            'run_dir': run_dir,
            'local_charge_cell': local_charge_path,
            'fit_npz': fit_path,
            'run_summary': summary_path if summary_path.exists() else None,
            'scalar_metrics': metrics_path if metrics_path.exists() else None,
        },
    }

payload = {
    'campaign_id': CAMPAIGN_ID,
    'runs': dict(sorted(runs.items(), key=lambda item: (item[1]['Ny'], item[1]['protocol']))),
    'skipped_runs': skipped_runs,
    'analysis_root': analysis_root,
}

inventory_df = pd.DataFrame([
    {
        'config_id': r['config_id'],
        'Nx': r['Nx'],
        'Ny': r['Ny'],
        'protocol': r['protocol'],
        'samples_actual': r['samples_actual'],
        'cycles': r['cycles'],
        'local_charge_shape': r['local_charge_cell'].shape,
        'fit_shape': r['entropy_slope'].shape,
    }
    for r in payload['runs'].values()
])
print(f'[loaded runs] {len(payload["runs"])}')
print(f'[skipped runs] {len(skipped_runs)}')
display(inventory_df)
display(pd.DataFrame(skipped_runs))

## 2. Per-Run Preview

Preview the scalar inputs used for the characterization.

In [ ]:
summary_rows = []
for config_id, record in payload['runs'].items():
    display(Markdown(f'### `{config_id}`'))
    info = {
        'protocol': record['protocol'],
        'Nx': record['Nx'],
        'Ny': record['Ny'],
        'samples_actual': record['samples_actual'],
        'cycles': record['cycles'],
        'q_pct_min': float(np.nanmin(record['q_pct'])),
        'q_pct_max': float(np.nanmax(record['q_pct'])),
        'entropy_slope_mean': float(np.nanmean(record['entropy_slope'])),
        'correlator_exponent_mean': float(np.nanmean(record['correlator_exponent'])),
    }
    summary_rows.append({'config_id': config_id, **info})
    print(json.dumps(info, indent=2))
    if not record['metrics_df'].empty:
        display(record['metrics_df'].head())

all_run_summary_df = pd.DataFrame(summary_rows)
display(Markdown('## All-Run Summary'))
display(all_run_summary_df)

## 3. Analysis Helpers

These cells define shared styling, video, summary, rolling-average, and scatter helpers.

In [ ]:
RUN_PERFECT_HISTOGRAM_VIDEOS = True
RUN_SPATIAL_CHARGE_HEATMAP_VIDEOS = True
RUN_PERFECT_ROLLING_FIGURES = True
RUN_PERFECT_SCATTER_FIGURES = True
RUN_POSTSELECT_FIGURES = True
WRITE_TABLES = True
WRITE_MANIFEST = True
DISPLAY_ARTIFACTS = True
FORCE_REGENERATE_VIDEOS = False
FORCE_REGENERATE_STATIC_FIGURES = True

PNG_DPI = 300
VIDEO_DPI = 200
FPS = 6
FRAME_REPEAT = 3
HISTOGRAM_START_CYCLE_EXCLUSIVE = 5
ENTROPY_HISTOGRAM_BINS = np.linspace(0.0, 2.0, 33)
VIDEO_FIGSIZE = (10.125, 3.6)
HEATMAP_VIDEO_FIGSIZE = (10.125, 3.75)
PLOT_FIGSIZE = (3.375, 2.4)
SCATTER_FIGSIZE = (6.75, 2.7)


def _ensure_cmu_sans_serif():
    target_name = 'CMU Sans Serif'
    if target_name in {entry.name for entry in font_manager.fontManager.ttflist}:
        return True
    for extension in ('ttf', 'otf'):
        for font_path in font_manager.findSystemFonts(fontext=extension):
            lower = Path(font_path).name.lower()
            if 'cmunss' not in lower and 'cmu' not in lower:
                continue
            try:
                font_manager.fontManager.addfont(font_path)
            except RuntimeError:
                pass
    return target_name in {entry.name for entry in font_manager.fontManager.ttflist}


def _preferred_font_family():
    return ['CMU Sans Serif', 'DejaVu Sans'] if _ensure_cmu_sans_serif() else ['DejaVu Sans']


def style_context():
    return {
        'font.family': _preferred_font_family(),
        'font.size': 8,
        'axes.titlesize': 8,
        'axes.labelsize': 8,
        'xtick.labelsize': 8,
        'ytick.labelsize': 8,
        'legend.fontsize': 8,
        'figure.dpi': PNG_DPI,
        'savefig.dpi': PNG_DPI,
    }


def configure_ffmpeg():
    ffmpeg_path = shutil.which('ffmpeg')
    if ffmpeg_path is None and imageio_ffmpeg is not None:
        ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
    if ffmpeg_path is None:
        raise RuntimeError('MP4 export needs ffmpeg. Install ffmpeg or imageio_ffmpeg.')
    plt.rcParams['animation.ffmpeg_path'] = ffmpeg_path
    return ffmpeg_path


def even_canvas_figsize(figsize, dpi):
    width_px = max(2, int(round(figsize[0] * dpi)))
    height_px = max(2, int(round(figsize[1] * dpi)))
    if width_px % 2:
        width_px += 1
    if height_px % 2:
        height_px += 1
    return (width_px / dpi, height_px / dpi)


def rolling_mean(values, window=ROLLING_WINDOW):
    return pd.Series(np.asarray(values, dtype=np.float64)).rolling(
        window=window, center=ROLLING_CENTER, min_periods=ROLLING_MIN_PERIODS
    ).mean().to_numpy(dtype=np.float64)


def protocol_records(protocol):
    records = [r for r in payload['runs'].values() if r['protocol'] == protocol and r['Ny'] in TARGET_NY]
    return sorted(records, key=lambda r: r['Ny'])


perfect_records = protocol_records('perfect_correction')
postselect_records = protocol_records('postselect')
print('[perfect records]', [r['config_id'] for r in perfect_records])
print('[postselect records]', [r['config_id'] for r in postselect_records])
if len(perfect_records) != 3 or len(postselect_records) != 3:
    raise ValueError('Expected three perfect-correction and three postselect records.')


def sample_rms(values, target=0.0):
    return np.sqrt(np.nanmean((np.asarray(values, dtype=np.float64) - target) ** 2, axis=0))


def sample_mean(values):
    return np.nanmean(np.asarray(values, dtype=np.float64), axis=0)


def build_cycle_summary(records, protocol_label):
    rows = []
    for record in records:
        q_pct = record['q_pct']
        entropy = record['entropy_slope']
        corr = record['correlator_exponent']
        for idx, cycle_label in enumerate(record['cycle_labels']):
            rows.append({
                'config_id': record['config_id'],
                'protocol': protocol_label,
                'Nx': record['Nx'],
                'Ny': record['Ny'],
                'cycle_label': int(cycle_label),
                'q_pct_mean': float(np.nanmean(q_pct[:, idx])),
                'q_pct_rms': float(np.sqrt(np.nanmean(q_pct[:, idx] ** 2))),
                'entropy_slope_mean': float(np.nanmean(entropy[:, idx])),
                'entropy_slope_rms_dev_from_1_over_3': float(np.sqrt(np.nanmean((entropy[:, idx] - ENTROPY_TARGET) ** 2))),
                'correlator_exponent_mean': float(np.nanmean(corr[:, idx])),
                'correlator_exponent_rms_dev_from_2': float(np.sqrt(np.nanmean((corr[:, idx] - CORRELATOR_TARGET) ** 2))),
            })
    return pd.DataFrame(rows)


def build_scatter_table(records, protocol_label):
    rows = []
    for record in records:
        mask = record['cycle_labels'] >= SCATTER_MIN_CYCLE
        cycle_labels = record['cycle_labels'][mask]
        for sample_pos, sample_index in enumerate(record['sample_indices']):
            for local_cycle_pos, cycle_label in enumerate(cycle_labels):
                cycle_pos = np.where(record['cycle_labels'] == cycle_label)[0][0]
                rows.append({
                    'config_id': record['config_id'],
                    'protocol': protocol_label,
                    'Nx': record['Nx'],
                    'Ny': record['Ny'],
                    'sample_index': int(sample_index),
                    'cycle_label': int(cycle_label),
                    'q_pct': float(record['q_pct'][sample_pos, cycle_pos]),
                    'entropy_slope': float(record['entropy_slope'][sample_pos, cycle_pos]),
                    'entropy_r2': float(record['entropy_r2'][sample_pos, cycle_pos]),
                    'correlator_exponent': float(record['correlator_exponent'][sample_pos, cycle_pos]),
                    'correlator_r2': float(record['correlator_r2'][sample_pos, cycle_pos]),
                })
    return pd.DataFrame(rows)


perfect_cycle_summary_df = build_cycle_summary(perfect_records, 'perfect_correction')
postselect_cycle_summary_df = build_cycle_summary(postselect_records, 'postselect')
perfect_scatter_df = build_scatter_table(perfect_records, 'perfect_correction')
postselect_scatter_df = build_scatter_table(postselect_records, 'postselect')

display(Markdown('### Perfect-Correction Cycle Summary'))
display(perfect_cycle_summary_df.head())
display(Markdown('### Postselection Cycle Summary'))
display(postselect_cycle_summary_df.head())
display(Markdown('### Perfect-Correction Scatter Table Preview'))
display(perfect_scatter_df.head())


## Shared Data/Display Utilities

These cells do not create figures. Each output cell below contains its own `plt`/`FuncAnimation` plotting code directly so titles, labels, legends, limits, scales, bins, and colors can be changed in place.


In [ ]:
def display_image_if_exists(title, path):
    if path is None:
        return
    path = Path(path)
    if path.exists():
        display(Markdown(f'### {title}'))
        display(Image(filename=str(path)))
    else:
        print(f'[missing image] {title}: {path}')


def display_video_if_exists(title, path):
    if path is None:
        return
    path = Path(path)
    if path.exists():
        display(Markdown(f'### {title}'))
        display(Video(filename=str(path), embed=False, html_attributes='controls preload="metadata" style="width:100%; max-width:1100px;"'))
    else:
        print(f'[missing video] {title}: {path}')


def _artifact_ready(*paths):
    return all(Path(path).exists() and Path(path).stat().st_size > 1024 for path in paths)


def _histogram_cycle_indices(record):
    indices = np.flatnonzero(record['cycle_labels'] > HISTOGRAM_START_CYCLE_EXCLUSIVE)
    if indices.size == 0:
        raise ValueError(f"{record['config_id']} has no cycles greater than {HISTOGRAM_START_CYCLE_EXCLUSIVE}")
    return indices


def _histogram_displayed_cycle_labels(records):
    return [record['cycle_labels'][_histogram_cycle_indices(record)].astype(int).tolist() for record in records]


def existing_histogram_video_artifact(records, video_path, final_frame_path):
    return {
        'video_path': Path(video_path),
        'final_frame_path': Path(final_frame_path),
        'ffmpeg_path': configure_ffmpeg(),
        'total_frames': max(len(_histogram_cycle_indices(record)) for record in records) * FRAME_REPEAT,
        'start_cycle_exclusive': HISTOGRAM_START_CYCLE_EXCLUSIVE,
        'displayed_cycle_labels': _histogram_displayed_cycle_labels(records),
    }


def existing_heatmap_video_artifact(records, video_path, final_frame_path):
    return {
        'video_path': Path(video_path),
        'final_frame_path': Path(final_frame_path),
        'ffmpeg_path': configure_ffmpeg(),
        'total_frames': max(int(record['cycles']) for record in records) * FRAME_REPEAT,
        'frames_per_cycle': FRAME_REPEAT,
    }


# Perfect-Correction Characterization


## Perfect-Correction Charge Histogram Video


In [ ]:
ffmpeg_path = configure_ffmpeg()
selected_indices = [_histogram_cycle_indices(record) for record in perfect_records]
pooled = np.concatenate([
    np.asarray(record['q_pct'][:, indices], dtype=np.float64).ravel()
    for record, indices in zip(perfect_records, selected_indices)
])
finite = pooled[np.isfinite(pooled)]
if finite.size == 0:
    raise ValueError(f'No finite data for q_pct')
_bins_spec = np.linspace(-4.0, 4.0, 33)
edges = np.histogram_bin_edges(finite, bins=32) if isinstance(_bins_spec, str) and _bins_spec == 'auto' else np.asarray(_bins_spec, dtype=np.float64)
bin_centers = 0.5 * (edges[:-1] + edges[1:])
bin_width = float(edges[1] - edges[0])
histograms = [
    np.stack([np.histogram(record['q_pct'][:, cycle_idx], bins=edges)[0] for cycle_idx in indices])
    for record, indices in zip(perfect_records, selected_indices)
]
max_count = max(float(np.max(histogram)) for histogram in histograms)
max_video_cycles = max(len(indices) for indices in selected_indices)
total_frames = max_video_cycles * FRAME_REPEAT

with plt.rc_context(style_context()):
    fig, axes = plt.subplots(1, len(perfect_records), figsize=even_canvas_figsize(VIDEO_FIGSIZE, VIDEO_DPI), sharey=True, constrained_layout=True)
    axes = np.atleast_1d(axes)
    bar_sets = []
    mean_lines = []
    for ax, record, indices, histogram in zip(axes, perfect_records, selected_indices, histograms):
        first_cycle_idx = int(indices[0])
        bars = ax.bar(bin_centers, histogram[0], width=0.92 * bin_width, color='#2b6cb0', edgecolor='#1a365d', linewidth=0.35)
        mean_value = float(np.nanmean(record['q_pct'][:, first_cycle_idx]))
        mean_line = ax.axvline(mean_value, color='#c53030', linestyle='--', linewidth=1.2)
        ax.set_xlabel(r'$100\,[2(\sum C_{cell}-N_xN_y)]/(2N_xN_y)$ (%)')
        ax.set_ylabel('sample count')
        ax.set_xlim(*(((-4.0, 4.0)) if ((-4.0, 4.0)) is not None else (float(edges[0]), float(edges[-1]))))
        ax.set_ylim(*(0, max(1.0, max_count * 1.08)))
        ax.set_xscale('linear')
        ax.set_yscale('linear')
        ax.grid(axis='y', alpha=0.25, linewidth=0.4)
        ax.set_box_aspect(0.9)
        bar_sets.append(bars)
        mean_lines.append(mean_line)
    fig.suptitle('Perfect correction: sample-to-sample centered charge percent deviation')

    def update(frame_idx):
        artists = []
        local_video_cycle = frame_idx // FRAME_REPEAT
        for ax, record, indices, histogram, bars, mean_line in zip(axes, perfect_records, selected_indices, histograms, bar_sets, mean_lines):
            selected_pos = min(local_video_cycle, len(indices) - 1)
            cycle_idx = int(indices[selected_pos])
            counts = histogram[selected_pos]
            for bar, height in zip(bars, counts):
                bar.set_height(float(height))
                artists.append(bar)
            values = record['q_pct'][:, cycle_idx]
            mean_value = float(np.nanmean(values))
            mean_line.set_xdata([mean_value, mean_value])
            artists.append(mean_line)
            ax.set_title((
                f"N{record['Nx']}x{record['Ny']} cycle {int(record['cycle_labels'][cycle_idx])}\n"
                f"Mean_raw={np.nanmean(record['q_raw'][:, cycle_idx]):.4f}; Mean_%={np.nanmean(values):.4f}\n"
                f"Var_raw={np.nanvar(record['q_raw'][:, cycle_idx]):.4f}; Var_%={np.nanvar(values):.5f}"
            ), pad=8)
            artists.append(ax.title)
        return artists

    update(0)
    writer = FFMpegWriter(fps=FPS, codec='libx264', bitrate=2200, extra_args=['-pix_fmt', 'yuv420p', '-movflags', '+faststart'])
    animation = FuncAnimation(fig, update, frames=total_frames, interval=1000 / FPS, blit=False)
    animation.save((figure_root / 'perfect_centered_charge_percent_histogram_comparison.mp4'), writer=writer, dpi=VIDEO_DPI)
    update(total_frames - 1)
    fig.savefig((figure_root / 'perfect_centered_charge_percent_histogram_final_frame.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)


display_video_if_exists('Perfect: centered charge histogram', figure_root / 'perfect_centered_charge_percent_histogram_comparison.mp4')
display_image_if_exists('Perfect: centered charge histogram final frame', figure_root / 'perfect_centered_charge_percent_histogram_final_frame.png')

## Perfect-Correction Entropy-Slope Histogram Video


In [ ]:
for record in perfect_records:
    record['entropy_slope_values'] = record['entropy_slope']

if RUN_PERFECT_HISTOGRAM_VIDEOS and (FORCE_REGENERATE_VIDEOS or not _artifact_ready((figure_root / 'perfect_entropy_slope_histogram_comparison.mp4'), (figure_root / 'perfect_entropy_slope_histogram_final_frame.png'))):
    ffmpeg_path = configure_ffmpeg()
    selected_indices = [_histogram_cycle_indices(record) for record in perfect_records]
    pooled = np.concatenate([
        np.asarray(record['entropy_slope_values'][:, indices], dtype=np.float64).ravel()
        for record, indices in zip(perfect_records, selected_indices)
    ])
    finite = pooled[np.isfinite(pooled)]
    if finite.size == 0:
        raise ValueError(f'No finite data for entropy_slope_values')
    edges = np.histogram_bin_edges(finite, bins=32) if isinstance(ENTROPY_HISTOGRAM_BINS, str) and ENTROPY_HISTOGRAM_BINS == 'auto' else np.asarray(ENTROPY_HISTOGRAM_BINS, dtype=np.float64)
    bin_centers = 0.5 * (edges[:-1] + edges[1:])
    bin_width = float(edges[1] - edges[0])
    histograms = [
        np.stack([np.histogram(record['entropy_slope_values'][:, cycle_idx], bins=edges)[0] for cycle_idx in indices])
        for record, indices in zip(perfect_records, selected_indices)
    ]
    max_count = max(float(np.max(histogram)) for histogram in histograms)
    max_video_cycles = max(len(indices) for indices in selected_indices)
    total_frames = max_video_cycles * FRAME_REPEAT

    with plt.rc_context(style_context()):
        fig, axes = plt.subplots(1, len(perfect_records), figsize=even_canvas_figsize(VIDEO_FIGSIZE, VIDEO_DPI), sharey=True, constrained_layout=True)
        axes = np.atleast_1d(axes)
        bar_sets = []
        mean_lines = []
        for ax, record, indices, histogram in zip(axes, perfect_records, selected_indices, histograms):
            first_cycle_idx = int(indices[0])
            bars = ax.bar(bin_centers, histogram[0], width=0.92 * bin_width, color='#2b6cb0', edgecolor='#1a365d', linewidth=0.35)
            mean_value = float(np.nanmean(record['entropy_slope_values'][:, first_cycle_idx]))
            mean_line = ax.axvline(mean_value, color='#c53030', linestyle='--', linewidth=1.2)
            ax.set_xlabel('entropy slope')
            ax.set_ylabel('sample count')
            ax.set_xlim(*(((0.0, 2.0)) if ((0.0, 2.0)) is not None else (float(edges[0]), float(edges[-1]))))
            ax.set_ylim(*(0, max(1.0, max_count * 1.08)))
            ax.set_xscale('linear')
            ax.set_yscale('linear')
            ax.grid(axis='y', alpha=0.25, linewidth=0.4)
            ax.set_box_aspect(0.9)
            bar_sets.append(bars)
            mean_lines.append(mean_line)
        fig.suptitle('Perfect correction: entropy-slope sample histogram')

        def update(frame_idx):
            artists = []
            local_video_cycle = frame_idx // FRAME_REPEAT
            for ax, record, indices, histogram, bars, mean_line in zip(axes, perfect_records, selected_indices, histograms, bar_sets, mean_lines):
                selected_pos = min(local_video_cycle, len(indices) - 1)
                cycle_idx = int(indices[selected_pos])
                counts = histogram[selected_pos]
                for bar, height in zip(bars, counts):
                    bar.set_height(float(height))
                    artists.append(bar)
                values = record['entropy_slope_values'][:, cycle_idx]
                mean_value = float(np.nanmean(values))
                mean_line.set_xdata([mean_value, mean_value])
                artists.append(mean_line)
                ax.set_title((
                    f"N{record['Nx']}x{record['Ny']} cycle {int(record['cycle_labels'][cycle_idx])}\n"
                    f"Mean_slope={np.nanmean(values):.4f}; Target=1/3\n"
                    f"RMS_dev={np.sqrt(np.nanmean((values - ENTROPY_TARGET) ** 2)):.5f}; Var={np.nanvar(values):.5f}"
                ), pad=8)
                artists.append(ax.title)
            return artists

        update(0)
        writer = FFMpegWriter(fps=FPS, codec='libx264', bitrate=2200, extra_args=['-pix_fmt', 'yuv420p', '-movflags', '+faststart'])
        animation = FuncAnimation(fig, update, frames=total_frames, interval=1000 / FPS, blit=False)
        animation.save((figure_root / 'perfect_entropy_slope_histogram_comparison.mp4'), writer=writer, dpi=VIDEO_DPI)
        update(total_frames - 1)
        fig.savefig((figure_root / 'perfect_entropy_slope_histogram_final_frame.png'), dpi=PNG_DPI, bbox_inches='tight')
        plt.close(fig)


display_video_if_exists('Perfect: entropy-slope histogram', figure_root / 'perfect_entropy_slope_histogram_comparison.mp4')
display_image_if_exists('Perfect: entropy-slope histogram final frame', figure_root / 'perfect_entropy_slope_histogram_final_frame.png')

## Perfect-Correction Final Entropy-Slope Deviation Histogram


In [ ]:
centers = 0.5 * (edges[:-1] + edges[1:])
width = float(edges[1] - edges[0])
final_indices = [int(record['cycles']) - 1 for record in perfect_records]
values_by_record = [record['entropy_slope'][:, cycle_idx] - ENTROPY_TARGET for record, cycle_idx in zip(perfect_records, final_indices)]
hists = [np.histogram(values[np.isfinite(values)], bins=edges)[0] for values in values_by_record]
max_count = max(float(np.max(hist)) for hist in hists)

with plt.rc_context(style_context()):
    fig, axes = plt.subplots(1, len(perfect_records), figsize=even_canvas_figsize(VIDEO_FIGSIZE, PNG_DPI), sharey=True, constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, record, cycle_idx, values, hist in zip(axes, perfect_records, final_indices, values_by_record, hists):
        ax.bar(centers, hist, width=0.92 * width, color='#2b6cb0', edgecolor='#1a365d', linewidth=0.35)
        mean_dev = float(np.nanmean(values))
        ax.axvline(0.0, color='#222222', linestyle=':', linewidth=1.0)
        ax.axvline(mean_dev, color='#c53030', linestyle='--', linewidth=1.2)
        ax.set_xlabel(r'entropy slope $-1/3$')
        ax.set_ylabel('sample count')
        ax.set_xlim(*((-0.1, 0.1)))
        ax.set_ylim(0, max(1.0, max_count * 1.08))
        ax.set_xscale('linear')
        ax.set_yscale('linear')
        ax.grid(axis='y', alpha=0.25, linewidth=0.4)
        ax.set_box_aspect(0.9)
        ax.set_title(
            f"N{record['Nx']}x{record['Ny']} cycle {int(record['cycle_labels'][cycle_idx])}\n"
            f"Mean_dev={mean_dev:.5f}; RMS_dev={np.sqrt(np.nanmean(values ** 2)):.5f}\n"
            f"Var_dev={np.nanvar(values):.5f}",
            pad=8,
        )
    fig.suptitle('Perfect correction: final entropy-slope deviation from 1/3')
    fig.savefig((figure_root / 'perfect_final_entropy_slope_deviation_histogram.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)

display_image_if_exists('Perfect: final entropy slope minus 1/3 histogram', (figure_root / 'perfect_final_entropy_slope_deviation_histogram.png'))

## Perfect-Correction Correlator-Exponent Histogram Video


In [ ]:
for record in perfect_records:
    record['correlator_exponent_values'] = record['correlator_exponent']

if RUN_PERFECT_HISTOGRAM_VIDEOS and (FORCE_REGENERATE_VIDEOS or not _artifact_ready((figure_root / 'perfect_correlator_exponent_histogram_comparison.mp4'), (figure_root / 'perfect_correlator_exponent_histogram_final_frame.png'))):
    ffmpeg_path = configure_ffmpeg()
    selected_indices = [_histogram_cycle_indices(record) for record in perfect_records]
    pooled = np.concatenate([
        np.asarray(record['correlator_exponent_values'][:, indices], dtype=np.float64).ravel()
        for record, indices in zip(perfect_records, selected_indices)
    ])
    finite = pooled[np.isfinite(pooled)]
    if finite.size == 0:
        raise ValueError(f'No finite data for correlator_exponent_values')
    edges = np.histogram_bin_edges(finite, bins=32) if True else np.asarray('auto', dtype=np.float64)
    bin_centers = 0.5 * (edges[:-1] + edges[1:])
    bin_width = float(edges[1] - edges[0])
    histograms = [
        np.stack([np.histogram(record['correlator_exponent_values'][:, cycle_idx], bins=edges)[0] for cycle_idx in indices])
        for record, indices in zip(perfect_records, selected_indices)
    ]
    max_count = max(float(np.max(histogram)) for histogram in histograms)
    max_video_cycles = max(len(indices) for indices in selected_indices)
    total_frames = max_video_cycles * FRAME_REPEAT

    with plt.rc_context(style_context()):
        fig, axes = plt.subplots(1, len(perfect_records), figsize=even_canvas_figsize(VIDEO_FIGSIZE, VIDEO_DPI), sharey=True, constrained_layout=True)
        axes = np.atleast_1d(axes)
        bar_sets = []
        mean_lines = []
        for ax, record, indices, histogram in zip(axes, perfect_records, selected_indices, histograms):
            first_cycle_idx = int(indices[0])
            bars = ax.bar(bin_centers, histogram[0], width=0.92 * bin_width, color='#2b6cb0', edgecolor='#1a365d', linewidth=0.35)
            mean_value = float(np.nanmean(record['correlator_exponent_values'][:, first_cycle_idx]))
            mean_line = ax.axvline(mean_value, color='#c53030', linestyle='--', linewidth=1.2)
            ax.set_xlabel('abs(correlator fitted slope)')
            ax.set_ylabel('sample count')
            ax.set_xlim(*(float(edges[0], float(edges[-1]))))
            ax.set_ylim(*(0, max(1.0, max_count * 1.08)))
            ax.set_xscale('linear')
            ax.set_yscale('linear')
            ax.grid(axis='y', alpha=0.25, linewidth=0.4)
            ax.set_box_aspect(0.9)
            bar_sets.append(bars)
            mean_lines.append(mean_line)
        fig.suptitle('Perfect correction: correlator-exponent sample histogram')

        def update(frame_idx):
            artists = []
            local_video_cycle = frame_idx // FRAME_REPEAT
            for ax, record, indices, histogram, bars, mean_line in zip(axes, perfect_records, selected_indices, histograms, bar_sets, mean_lines):
                selected_pos = min(local_video_cycle, len(indices) - 1)
                cycle_idx = int(indices[selected_pos])
                counts = histogram[selected_pos]
                for bar, height in zip(bars, counts):
                    bar.set_height(float(height))
                    artists.append(bar)
                values = record['correlator_exponent_values'][:, cycle_idx]
                mean_value = float(np.nanmean(values))
                mean_line.set_xdata([mean_value, mean_value])
                artists.append(mean_line)
                ax.set_title((
                    f"N{record['Nx']}x{record['Ny']} cycle {int(record['cycle_labels'][cycle_idx])}\n"
                    f"Mean_|slope|={np.nanmean(values):.4f}; Target=2\n"
                    f"RMS_dev={np.sqrt(np.nanmean((values - CORRELATOR_TARGET) ** 2)):.5f}; Var={np.nanvar(values):.5f}"
                ), pad=8)
                artists.append(ax.title)
            return artists

        update(0)
        writer = FFMpegWriter(fps=FPS, codec='libx264', bitrate=2200, extra_args=['-pix_fmt', 'yuv420p', '-movflags', '+faststart'])
        animation = FuncAnimation(fig, update, frames=total_frames, interval=1000 / FPS, blit=False)
        animation.save((figure_root / 'perfect_correlator_exponent_histogram_comparison.mp4'), writer=writer, dpi=VIDEO_DPI)
        update(total_frames - 1)
        fig.savefig((figure_root / 'perfect_correlator_exponent_histogram_final_frame.png'), dpi=PNG_DPI, bbox_inches='tight')
        plt.close(fig)


display_video_if_exists('Perfect: correlator-exponent histogram', figure_root / 'perfect_correlator_exponent_histogram_comparison.mp4')
display_image_if_exists('Perfect: correlator-exponent histogram final frame', figure_root / 'perfect_correlator_exponent_histogram_final_frame.png')

## Perfect-Correction Spatial Charge Heatmap Video


In [ ]:
ffmpeg_path = configure_ffmpeg()
averaged_charge = [np.nanmean(record['local_charge_cell'], axis=0) for record in perfect_records]
finite = np.concatenate([arr[np.isfinite(arr)].ravel() for arr in averaged_charge])
vmin = float(np.nanpercentile(finite, ((1.0, 99.0))[0]))
vmax = float(np.nanpercentile(finite, ((1.0, 99.0))[1]))
if not np.isfinite(vmin) or not np.isfinite(vmax) or np.isclose(vmin, vmax):
    vmin = float(np.nanmin(finite))
    vmax = float(np.nanmax(finite))
if np.isclose(vmin, vmax):
    vmin -= 0.5
    vmax += 0.5
if vmin < 1.0 < vmax:
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=1.0, vmax=vmax)
else:
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

max_cycles = max(int(record['cycles']) for record in perfect_records)
total_frames = max_cycles * FRAME_REPEAT
with plt.rc_context(style_context()):
    fig, axes = plt.subplots(1, len(perfect_records), figsize=even_canvas_figsize(HEATMAP_VIDEO_FIGSIZE, VIDEO_DPI), constrained_layout=True)
    axes = np.atleast_1d(axes)
    images = []
    for ax, record, charge in zip(axes, perfect_records, averaged_charge):
        image = ax.imshow(charge[0], origin='lower', aspect='auto', cmap='coolwarm', norm=norm, interpolation='nearest')
        ax.set_xlabel('y')
        ax.set_ylabel('x')
        ax.set_box_aspect(0.9)
        images.append(image)
    colorbar = fig.colorbar(images[-1], ax=axes, shrink=0.88, pad=0.02)
    colorbar.set_label('local charge per unit cell')
    fig.suptitle('Perfect correction: sample-averaged spatial local charge per unit cell')

    def update(frame_idx):
        artists = []
        video_cycle = frame_idx // FRAME_REPEAT
        for ax, record, charge, image in zip(axes, perfect_records, averaged_charge, images):
            cycle_idx = min(video_cycle, int(record['cycles']) - 1)
            image.set_data(charge[cycle_idx])
            artists.append(image)
            mean_charge = float(np.nanmean(charge[cycle_idx]))
            cycle_label = int(record['cycle_labels'][cycle_idx])
            ax.set_title(
                f"N{record['Nx']}x{record['Ny']} cycle {cycle_label}\n"
                f"{'sample averaged'}; mean={mean_charge:.6f}",
                pad=8,
            )
            artists.append(ax.title)
        return artists

    update(0)
    writer = FFMpegWriter(fps=FPS, codec='libx264', bitrate=2400, extra_args=['-pix_fmt', 'yuv420p', '-movflags', '+faststart'])
    animation = FuncAnimation(fig, update, frames=total_frames, interval=1000 / FPS, blit=False)
    animation.save((figure_root / 'perfect_sample_averaged_local_charge_heatmap_comparison.mp4'), writer=writer, dpi=VIDEO_DPI)
    update(total_frames - 1)
    fig.savefig((figure_root / 'perfect_sample_averaged_local_charge_heatmap_final_frame.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)


display_video_if_exists('Perfect: sample-averaged local charge heatmap', figure_root / 'perfect_sample_averaged_local_charge_heatmap_comparison.mp4')
display_image_if_exists('Perfect: sample-averaged local charge heatmap final frame', figure_root / 'perfect_sample_averaged_local_charge_heatmap_final_frame.png')

## Perfect-Correction Charge RMS Vs Cycle


In [ ]:
with plt.rc_context(style_context()):
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, constrained_layout=True)
    for config_id, group in perfect_cycle_summary_df.groupby('config_id', sort=False):
        group = group.sort_values('cycle_label')
        label = f"N{int(group['Nx'].iloc[0])}x{int(group['Ny'].iloc[0])}"
        cycles = group['cycle_label'].to_numpy(dtype=np.float64)
        values = group['q_pct_rms'].to_numpy(dtype=np.float64)
        raw_mask = np.isfinite(cycles) & np.isfinite(values)
        rolling_values = rolling_mean(values)
        rolling_mask = np.isfinite(cycles) & np.isfinite(rolling_values)
        if np.any(raw_mask):
            ax.plot(cycles[raw_mask], values[raw_mask], alpha=0.35, linewidth=1.0, label=f'{label}: raw')
        if np.any(rolling_mask):
            ax.plot(cycles[rolling_mask], rolling_values[rolling_mask], linewidth=1.6, label=f'{label}: rolling w={ROLLING_WINDOW}')
    ax.set_xlabel('cycle')
    ax.set_ylabel('RMS centered charge [%]')
    ax.set_title('Perfect correction: RMS centered charge [%]')
    ax.set_xscale('linear')
    ax.set_yscale('linear')
    ax.grid(True, which='major', alpha=0.25, linewidth=0.4)
    ax.legend(loc='best')
    fig.savefig((figure_root / 'perfect_correction_charge_percent_rms_vs_cycle_rolling.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)

display_image_if_exists('Perfect: charge rms vs cycle', (figure_root / 'perfect_correction_charge_percent_rms_vs_cycle_rolling.png'))

## Perfect-Correction Entropy-Slope RMS Deviation Vs Cycle Log-Log


In [ ]:
with plt.rc_context(style_context()):
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, constrained_layout=True)
    for config_id, group in perfect_cycle_summary_df.groupby('config_id', sort=False):
        group = group.sort_values('cycle_label')
        label = f"N{int(group['Nx'].iloc[0])}x{int(group['Ny'].iloc[0])}"
        cycles = group['cycle_label'].to_numpy(dtype=np.float64)
        values = group['entropy_slope_rms_dev_from_1_over_3'].to_numpy(dtype=np.float64)
        raw_mask = np.isfinite(cycles) & np.isfinite(values)
        raw_mask &= cycles > 0.0
        raw_mask &= values > 0.0
        rolling_values = rolling_mean(values)
        rolling_mask = np.isfinite(cycles) & np.isfinite(rolling_values)
        rolling_mask &= cycles > 0.0
        rolling_mask &= rolling_values > 0.0
        if np.any(raw_mask):
            ax.plot(cycles[raw_mask], values[raw_mask], alpha=0.35, linewidth=1.0, label=f'{label}: raw')
        if np.any(rolling_mask):
            ax.plot(cycles[rolling_mask], rolling_values[rolling_mask], linewidth=1.6, label=f'{label}: rolling w={ROLLING_WINDOW}')
    ax.set_xlabel('cycle')
    ax.set_ylabel('RMS($s_{S}-1/3$)')
    ax.set_title('Perfect correction: RMS($s_S-1/3$) log-log')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, which='both', alpha=0.25, linewidth=0.4)
    ax.legend(loc='best')
    fig.savefig((figure_root / 'perfect_correction_entropy_slope_rms_dev_vs_cycle_rolling.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)

display_image_if_exists('Perfect: entropy-slope rms deviation vs cycle log-log', (figure_root / 'perfect_correction_entropy_slope_rms_dev_vs_cycle_rolling.png'))

## Perfect-Correction Correlator-Exponent RMS Deviation Vs Cycle Log-Log


In [ ]:
with plt.rc_context(style_context()):
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, constrained_layout=True)
    for config_id, group in perfect_cycle_summary_df.groupby('config_id', sort=False):
        group = group.sort_values('cycle_label')
        label = f"N{int(group['Nx'].iloc[0])}x{int(group['Ny'].iloc[0])}"
        cycles = group['cycle_label'].to_numpy(dtype=np.float64)
        values = group['correlator_exponent_rms_dev_from_2'].to_numpy(dtype=np.float64)
        raw_mask = np.isfinite(cycles) & np.isfinite(values)
        raw_mask &= cycles > 0.0
        raw_mask &= values > 0.0
        rolling_values = rolling_mean(values)
        rolling_mask = np.isfinite(cycles) & np.isfinite(rolling_values)
        rolling_mask &= cycles > 0.0
        rolling_mask &= rolling_values > 0.0
        if np.any(raw_mask):
            ax.plot(cycles[raw_mask], values[raw_mask], alpha=0.35, linewidth=1.0, label=f'{label}: raw')
        if np.any(rolling_mask):
            ax.plot(cycles[rolling_mask], rolling_values[rolling_mask], linewidth=1.6, label=f'{label}: rolling w={ROLLING_WINDOW}')
    ax.set_xlabel('cycle')
    ax.set_ylabel('RMS($|s_C|-2$)')
    ax.set_title('Perfect correction: RMS($|s_C|-2$) log-log')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, which='both', alpha=0.25, linewidth=0.4)
    ax.legend(loc='best')
    fig.savefig((figure_root / 'perfect_correction_correlator_exponent_rms_dev_vs_cycle_rolling.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)

display_image_if_exists('Perfect: correlator-exponent rms deviation vs cycle log-log', (figure_root / 'perfect_correction_correlator_exponent_rms_dev_vs_cycle_rolling.png'))

## Perfect-Correction Entropy Slope/R2 Vs Charge Scatter


In [ ]:
with plt.rc_context(style_context()):
    fig, axes = plt.subplots(1, 2, figsize=SCATTER_FIGSIZE, constrained_layout=True)
    for ny, group in perfect_scatter_df.groupby('Ny', sort=True):
        axes[0].scatter(group['q_pct'], group['entropy_slope'], s=4, alpha=0.28, label=f'Ny={ny}', rasterized=True)
        axes[1].scatter(group['q_pct'], group['entropy_r2'], s=4, alpha=0.28, label=f'Ny={ny}', rasterized=True)
    axes[0].set_xlabel('centered charge deviation [%]')
    axes[0].set_ylabel('entropy slope')
    axes[0].set_title((f'Entropy slope vs charge, cycles >= {SCATTER_MIN_CYCLE}'))
    axes[0].set_xscale('linear')
    axes[0].set_yscale('linear')
    axes[0].grid(alpha=0.25, linewidth=0.4)
    axes[1].set_xlabel('centered charge deviation [%]')
    axes[1].set_ylabel(r'$R^2$')
    axes[1].set_title(r'fit $R^2$')
    axes[1].set_xscale('linear')
    axes[1].set_yscale('linear')
    axes[1].grid(alpha=0.25, linewidth=0.4)
    axes[1].legend(loc='best')
    fig.savefig((figure_root / 'perfect_entropy_slope_and_r2_vs_charge_scatter.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)

display_image_if_exists('Perfect: entropy slope and R2 vs charge', (figure_root / 'perfect_entropy_slope_and_r2_vs_charge_scatter.png'))

## Perfect-Correction Correlator Exponent/R2 Vs Charge Scatter


In [ ]:
with plt.rc_context(style_context()):
    fig, axes = plt.subplots(1, 2, figsize=SCATTER_FIGSIZE, constrained_layout=True)
    for ny, group in perfect_scatter_df.groupby('Ny', sort=True):
        axes[0].scatter(group['q_pct'], group['correlator_exponent'], s=4, alpha=0.28, label=f'Ny={ny}', rasterized=True)
        axes[1].scatter(group['q_pct'], group['correlator_r2'], s=4, alpha=0.28, label=f'Ny={ny}', rasterized=True)
    axes[0].set_xlabel('centered charge deviation [%]')
    axes[0].set_ylabel('abs(correlator slope)')
    axes[0].set_title((f'Correlator exponent vs charge, cycles >= {SCATTER_MIN_CYCLE}'))
    axes[0].set_xscale('linear')
    axes[0].set_yscale('linear')
    axes[0].grid(alpha=0.25, linewidth=0.4)
    axes[1].set_xlabel('centered charge deviation [%]')
    axes[1].set_ylabel(r'$R^2$')
    axes[1].set_title(r'fit $R^2$')
    axes[1].set_xscale('linear')
    axes[1].set_yscale('linear')
    axes[1].grid(alpha=0.25, linewidth=0.4)
    axes[1].legend(loc='best')
    fig.savefig((figure_root / 'perfect_correlator_exponent_and_r2_vs_charge_scatter.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)

display_image_if_exists('Perfect: correlator exponent and R2 vs charge', (figure_root / 'perfect_correlator_exponent_and_r2_vs_charge_scatter.png'))

## Perfect-Correction Chern Number Histogram

Histogram of the real-space Chern number $\nu$ pooled over all samples at cycles $t \geq t_{\min}$.

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.4))
with plt.rc_context(style_context()):
    for record in perfect_records:
        df = record['metrics_df']
        if df.empty or 'real_space_chern' not in df.columns:
            continue
        vals = df.loc[df['cycle_label'] >= SCATTER_MIN_CYCLE, 'real_space_chern'].dropna().values
        ax.hist(vals, bins=30, alpha=0.6, label=rf"$N_y = {record['Ny']}$")
    for nu in (0, 1):
        ax.axvline(nu, color='k', lw=0.8, ls='--')
    ax.set_xlabel(r'$\nu$')
    ax.set_ylabel('Count')
    ax.set_title(r'Real-Space Chern Number — Perfect Correction ($t \geq t_{\min}$)')
    ax.legend(loc='upper left')
    plt.tight_layout()
    out = figure_root / 'perfect_correction_chern_histogram.png'
    fig.savefig(out, dpi=PNG_DPI)
plt.show()
display_image_if_exists('Perfect-Correction Chern Histogram', out)

## Perfect-Correction Sample-Averaged Chern Number Vs Cycle

Sample-averaged real-space Chern number $\overline{\nu}(t)$ as a function of circuit cycle for each system size.

In [ ]:
with plt.rc_context(style_context()):
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, constrained_layout=True)
    for record in perfect_records:
        df = record['metrics_df']
        if df.empty or 'real_space_chern' not in df.columns:
            continue
        label = f"N{record['Nx']}x{record['Ny']}"
        grp = df.groupby('cycle_label')['real_space_chern'].mean().sort_index()
        cycles = grp.index.to_numpy(dtype=np.float64)
        values = grp.to_numpy(dtype=np.float64)
        mask = np.isfinite(cycles) & np.isfinite(values)
        rolling = rolling_mean(values)
        rmask = np.isfinite(rolling)
        if np.any(mask):
            ax.plot(cycles[mask], values[mask], alpha=0.35, linewidth=1.0, label=f'{label}: raw')
        if np.any(rmask):
            ax.plot(cycles[rmask], rolling[rmask], linewidth=1.6, label=f'{label}: rolling w={ROLLING_WINDOW}')
    ax.axhline(1, color='k', lw=0.8, ls='--', label=r'$\nu = 1$')
    ax.set_xlabel('cycle')
    ax.set_ylabel(r'$\overline{\nu}$')
    ax.set_title('Perfect correction: sample-averaged Chern number vs cycle')
    ax.grid(True, which='major', alpha=0.25, linewidth=0.4)
    ax.legend(loc='best')
    out = figure_root / 'perfect_correction_chern_vs_cycle.png'
    fig.savefig(out, dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)
display_image_if_exists('Perfect: sample-averaged Chern vs cycle', out)

# Post-Selection Characterization

Post-selection has one trajectory per system size. These cells produce single-trajectory diagnostics, not sample statistics.


## Post-Selection Derived Plot Table


In [ ]:
postselect_plot_df = postselect_cycle_summary_df.copy()
postselect_plot_df['q_pct_rms'] = np.abs(postselect_plot_df['q_pct_mean'])
postselect_plot_df['entropy_slope_rms_dev_from_1_over_3'] = np.abs(
    postselect_plot_df['entropy_slope_mean'] - ENTROPY_TARGET
)
postselect_plot_df['correlator_exponent_rms_dev_from_2'] = np.abs(
    postselect_plot_df['correlator_exponent_mean'] - CORRELATOR_TARGET
)
display(postselect_plot_df.head())


## Post-Selection Spatial Charge Heatmap Video


In [ ]:
ffmpeg_path = configure_ffmpeg()
averaged_charge = [np.nanmean(record['local_charge_cell'], axis=0) for record in postselect_records]
finite = np.concatenate([arr[np.isfinite(arr)].ravel() for arr in averaged_charge])
vmin = float(np.nanpercentile(finite, ((1.0, 99.0))[0]))
vmax = float(np.nanpercentile(finite, ((1.0, 99.0))[1]))
if not np.isfinite(vmin) or not np.isfinite(vmax) or np.isclose(vmin, vmax):
    vmin = float(np.nanmin(finite))
    vmax = float(np.nanmax(finite))
if np.isclose(vmin, vmax):
    vmin -= 0.5
    vmax += 0.5
if vmin < 1.0 < vmax:
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=1.0, vmax=vmax)
else:
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

max_cycles = max(int(record['cycles']) for record in postselect_records)
total_frames = max_cycles * FRAME_REPEAT
with plt.rc_context(style_context()):
    fig, axes = plt.subplots(1, len(postselect_records), figsize=even_canvas_figsize(HEATMAP_VIDEO_FIGSIZE, VIDEO_DPI), constrained_layout=True)
    axes = np.atleast_1d(axes)
    images = []
    for ax, record, charge in zip(axes, postselect_records, averaged_charge):
        image = ax.imshow(charge[0], origin='lower', aspect='auto', cmap='coolwarm', norm=norm, interpolation='nearest')
        ax.set_xlabel('y')
        ax.set_ylabel('x')
        ax.set_box_aspect(0.9)
        images.append(image)
    colorbar = fig.colorbar(images[-1], ax=axes, shrink=0.88, pad=0.02)
    colorbar.set_label('local charge per unit cell')
    fig.suptitle('Postselection: single-trajectory spatial local charge per unit cell')

    def update(frame_idx):
        artists = []
        video_cycle = frame_idx // FRAME_REPEAT
        for ax, record, charge, image in zip(axes, postselect_records, averaged_charge, images):
            cycle_idx = min(video_cycle, int(record['cycles']) - 1)
            image.set_data(charge[cycle_idx])
            artists.append(image)
            mean_charge = float(np.nanmean(charge[cycle_idx]))
            cycle_label = int(record['cycle_labels'][cycle_idx])
            ax.set_title(
                f"N{record['Nx']}x{record['Ny']} cycle {cycle_label}\n"
                f"{'single trajectory'}; mean={mean_charge:.6f}",
                pad=8,
            )
            artists.append(ax.title)
        return artists

    update(0)
    writer = FFMpegWriter(fps=FPS, codec='libx264', bitrate=2400, extra_args=['-pix_fmt', 'yuv420p', '-movflags', '+faststart'])
    animation = FuncAnimation(fig, update, frames=total_frames, interval=1000 / FPS, blit=False)
    animation.save((figure_root / 'postselect_local_charge_heatmap_comparison.mp4'), writer=writer, dpi=VIDEO_DPI)
    update(total_frames - 1)
    fig.savefig((figure_root / 'postselect_local_charge_heatmap_final_frame.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)


display_video_if_exists('Postselection: local charge heatmap', figure_root / 'postselect_local_charge_heatmap_comparison.mp4')
display_image_if_exists('Postselection: local charge heatmap final frame', figure_root / 'postselect_local_charge_heatmap_final_frame.png')

## Post-Selection Absolute Charge Deviation Vs Cycle


In [ ]:
with plt.rc_context(style_context()):
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, constrained_layout=True)
    for config_id, group in postselect_plot_df.groupby('config_id', sort=False):
        group = group.sort_values('cycle_label')
        label = f"N{int(group['Nx'].iloc[0])}x{int(group['Ny'].iloc[0])}"
        cycles = group['cycle_label'].to_numpy(dtype=np.float64)
        values = group['q_pct_rms'].to_numpy(dtype=np.float64)
        raw_mask = np.isfinite(cycles) & np.isfinite(values)
        rolling_values = rolling_mean(values)
        rolling_mask = np.isfinite(cycles) & np.isfinite(rolling_values)
        if np.any(raw_mask):
            ax.plot(cycles[raw_mask], values[raw_mask], alpha=0.35, linewidth=1.0, label=f'{label}: raw')
        if np.any(rolling_mask):
            ax.plot(cycles[rolling_mask], rolling_values[rolling_mask], linewidth=1.6, label=f'{label}: rolling w={ROLLING_WINDOW}')
    ax.set_xlabel('cycle')
    ax.set_ylabel('abs centered charge [%]')
    ax.set_title('Postselection single trajectory: absolute centered charge [%]')
    ax.set_xscale('linear')
    ax.set_yscale('linear')
    ax.grid(True, which='major', alpha=0.25, linewidth=0.4)
    ax.legend(loc='best')
    fig.savefig((figure_root / 'postselect_charge_percent_rms_vs_cycle_rolling.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)

display_image_if_exists('Postselection: absolute charge deviation vs cycle', (figure_root / 'postselect_charge_percent_rms_vs_cycle_rolling.png'))

## Post-Selection Entropy-Slope Absolute Deviation Vs Cycle Log-Log


In [ ]:
with plt.rc_context(style_context()):
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, constrained_layout=True)
    for config_id, group in postselect_plot_df.groupby('config_id', sort=False):
        group = group.sort_values('cycle_label')
        label = f"N{int(group['Nx'].iloc[0])}x{int(group['Ny'].iloc[0])}"
        cycles = group['cycle_label'].to_numpy(dtype=np.float64)
        values = group['entropy_slope_rms_dev_from_1_over_3'].to_numpy(dtype=np.float64)
        raw_mask = np.isfinite(cycles) & np.isfinite(values)
        raw_mask &= cycles > 0.0
        raw_mask &= values > 0.0
        rolling_values = rolling_mean(values)
        rolling_mask = np.isfinite(cycles) & np.isfinite(rolling_values)
        rolling_mask &= cycles > 0.0
        rolling_mask &= rolling_values > 0.0
        if np.any(raw_mask):
            ax.plot(cycles[raw_mask], values[raw_mask], alpha=0.35, linewidth=1.0, label=f'{label}: raw')
        if np.any(rolling_mask):
            ax.plot(cycles[rolling_mask], rolling_values[rolling_mask], linewidth=1.6, label=f'{label}: rolling w={ROLLING_WINDOW}')
    ax.set_xlabel('cycle')
    ax.set_ylabel('$|s_{S}-1/3|$')
    ax.set_title('Postselection single trajectory: $|s_S-1/3|$ log-log')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, which='both', alpha=0.25, linewidth=0.4)
    ax.legend(loc='best')
    fig.savefig((figure_root / 'postselect_entropy_slope_rms_dev_vs_cycle_rolling.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)

display_image_if_exists('Postselection: entropy-slope absolute deviation vs cycle log-log', (figure_root / 'postselect_entropy_slope_rms_dev_vs_cycle_rolling.png'))

## Post-Selection Correlator-Exponent Absolute Deviation Vs Cycle Log-Log


In [ ]:
with plt.rc_context(style_context()):
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, constrained_layout=True)
    for config_id, group in postselect_plot_df.groupby('config_id', sort=False):
        group = group.sort_values('cycle_label')
        label = f"N{int(group['Nx'].iloc[0])}x{int(group['Ny'].iloc[0])}"
        cycles = group['cycle_label'].to_numpy(dtype=np.float64)
        values = group['correlator_exponent_rms_dev_from_2'].to_numpy(dtype=np.float64)
        raw_mask = np.isfinite(cycles) & np.isfinite(values)
        raw_mask &= cycles > 0.0
        raw_mask &= values > 0.0
        rolling_values = rolling_mean(values)
        rolling_mask = np.isfinite(cycles) & np.isfinite(rolling_values)
        rolling_mask &= cycles > 0.0
        rolling_mask &= rolling_values > 0.0
        if np.any(raw_mask):
            ax.plot(cycles[raw_mask], values[raw_mask], alpha=0.35, linewidth=1.0, label=f'{label}: raw')
        if np.any(rolling_mask):
            ax.plot(cycles[rolling_mask], rolling_values[rolling_mask], linewidth=1.6, label=f'{label}: rolling w={ROLLING_WINDOW}')
    ax.set_xlabel('cycle')
    ax.set_ylabel('$||s_C|-2|$')
    ax.set_title('Postselection single trajectory: $||s_C|-2|$ log-log')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, which='both', alpha=0.25, linewidth=0.4)
    ax.legend(loc='best')
    fig.savefig((figure_root / 'postselect_correlator_exponent_rms_dev_vs_cycle_rolling.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)

display_image_if_exists('Postselection: correlator-exponent absolute deviation vs cycle log-log', (figure_root / 'postselect_correlator_exponent_rms_dev_vs_cycle_rolling.png'))

## Post-Selection Entropy Slope/R2 Vs Charge Scatter


In [ ]:
with plt.rc_context(style_context()):
    fig, axes = plt.subplots(1, 2, figsize=SCATTER_FIGSIZE, constrained_layout=True)
    for ny, group in postselect_scatter_df.groupby('Ny', sort=True):
        axes[0].scatter(group['q_pct'], group['entropy_slope'], s=4, alpha=0.28, label=f'Ny={ny}', rasterized=True)
        axes[1].scatter(group['q_pct'], group['entropy_r2'], s=4, alpha=0.28, label=f'Ny={ny}', rasterized=True)
    axes[0].set_xlabel('centered charge deviation [%]')
    axes[0].set_ylabel('entropy slope')
    axes[0].set_title((f'Postselect entropy slope vs charge, cycles >= {SCATTER_MIN_CYCLE}'))
    axes[0].set_xscale('linear')
    axes[0].set_yscale('linear')
    axes[0].grid(alpha=0.25, linewidth=0.4)
    axes[1].set_xlabel('centered charge deviation [%]')
    axes[1].set_ylabel(r'$R^2$')
    axes[1].set_title(r'fit $R^2$')
    axes[1].set_xscale('linear')
    axes[1].set_yscale('linear')
    axes[1].grid(alpha=0.25, linewidth=0.4)
    axes[1].legend(loc='best')
    fig.savefig((figure_root / 'postselect_entropy_slope_and_r2_vs_charge_scatter.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)

display_image_if_exists('Postselection: entropy slope and R2 vs charge', (figure_root / 'postselect_entropy_slope_and_r2_vs_charge_scatter.png'))

## Post-Selection Correlator Exponent/R2 Vs Charge Scatter


In [ ]:
with plt.rc_context(style_context()):
    fig, axes = plt.subplots(1, 2, figsize=SCATTER_FIGSIZE, constrained_layout=True)
    for ny, group in postselect_scatter_df.groupby('Ny', sort=True):
        axes[0].scatter(group['q_pct'], group['correlator_exponent'], s=4, alpha=0.28, label=f'Ny={ny}', rasterized=True)
        axes[1].scatter(group['q_pct'], group['correlator_r2'], s=4, alpha=0.28, label=f'Ny={ny}', rasterized=True)
    axes[0].set_xlabel('centered charge deviation [%]')
    axes[0].set_ylabel('abs(correlator slope)')
    axes[0].set_title((f'Postselect correlator exponent vs charge, cycles >= {SCATTER_MIN_CYCLE}'))
    axes[0].set_xscale('linear')
    axes[0].set_yscale('linear')
    axes[0].grid(alpha=0.25, linewidth=0.4)
    axes[1].set_xlabel('centered charge deviation [%]')
    axes[1].set_ylabel(r'$R^2$')
    axes[1].set_title(r'fit $R^2$')
    axes[1].set_xscale('linear')
    axes[1].set_yscale('linear')
    axes[1].grid(alpha=0.25, linewidth=0.4)
    axes[1].legend(loc='best')
    fig.savefig((figure_root / 'postselect_correlator_exponent_and_r2_vs_charge_scatter.png'), dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)

display_image_if_exists('Postselection: correlator exponent and R2 vs charge', (figure_root / 'postselect_correlator_exponent_and_r2_vs_charge_scatter.png'))

## Post-Selection Chern Number Histogram

Histogram of the real-space Chern number $\nu$ for post-selected trajectories at cycles $t \geq t_{\min}$.

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.4))
with plt.rc_context(style_context()):
    for record in postselect_records:
        df = record['metrics_df']
        if df.empty or 'real_space_chern' not in df.columns:
            continue
        vals = df.loc[df['cycle_label'] >= SCATTER_MIN_CYCLE, 'real_space_chern'].dropna().values
        ax.hist(vals, bins=30, alpha=0.6, label=rf"$N_y = {record['Ny']}$")
    for nu in (0, 1):
        ax.axvline(nu, color='k', lw=0.8, ls='--')
    ax.set_xlabel(r'$\nu$')
    ax.set_ylabel('Count')
    ax.set_title(r'Real-Space Chern Number — Post-Selection ($t \geq t_{\min}$)')
    ax.legend(loc='upper left')
    plt.tight_layout()
    out = figure_root / 'postselect_chern_histogram.png'
    fig.savefig(out, dpi=PNG_DPI)
plt.show()
display_image_if_exists('Post-Selection Chern Histogram', out)

## Post-Selection Sample-Averaged Chern Number Vs Cycle

Sample-averaged real-space Chern number $\overline{\nu}(t)$ as a function of circuit cycle for each system size.

In [ ]:
with plt.rc_context(style_context()):
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, constrained_layout=True)
    for record in postselect_records:
        df = record['metrics_df']
        if df.empty or 'real_space_chern' not in df.columns:
            continue
        label = f"N{record['Nx']}x{record['Ny']}"
        grp = df.groupby('cycle_label')['real_space_chern'].mean().sort_index()
        cycles = grp.index.to_numpy(dtype=np.float64)
        values = grp.to_numpy(dtype=np.float64)
        mask = np.isfinite(cycles) & np.isfinite(values)
        rolling = rolling_mean(values)
        rmask = np.isfinite(rolling)
        if np.any(mask):
            ax.plot(cycles[mask], values[mask], alpha=0.35, linewidth=1.0, label=f'{label}: raw')
        if np.any(rmask):
            ax.plot(cycles[rmask], rolling[rmask], linewidth=1.6, label=f'{label}: rolling w={ROLLING_WINDOW}')
    ax.axhline(1, color='k', lw=0.8, ls='--', label=r'$\nu = 1$')
    ax.set_xlabel('cycle')
    ax.set_ylabel(r'$\overline{\nu}$')
    ax.set_title('Post-selection: sample-averaged Chern number vs cycle')
    ax.grid(True, which='major', alpha=0.25, linewidth=0.4)
    ax.legend(loc='best')
    out = figure_root / 'postselect_chern_vs_cycle.png'
    fig.savefig(out, dpi=PNG_DPI, bbox_inches='tight')
    plt.close(fig)
display_image_if_exists('Post-selection: sample-averaged Chern vs cycle', out)

# Tables And Manifest


In [ ]:
artifact_paths = {
    'figures': sorted(figure_root.glob('*.png')),
    'videos': sorted(figure_root.glob('*.mp4')),
}

table_paths = {}
if WRITE_TABLES:
    tables = {
        'perfect_cycle_summary': perfect_cycle_summary_df,
        'postselect_cycle_summary': postselect_cycle_summary_df,
        'perfect_scatter_cycles_ge20': perfect_scatter_df,
        'postselect_scatter_cycles_ge20': postselect_scatter_df,
    }
    for name, df in tables.items():
        path = table_root / f'{name}.csv'
        df.to_csv(path, index=False)
        table_paths[name] = path
        print(f'[table {name}] {path} rows={len(df)}')


def _path_to_string(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {key: _path_to_string(value) for key, value in obj.items()}
    if isinstance(obj, list):
        return [_path_to_string(value) for value in obj]
    return obj

if WRITE_MANIFEST:
    manifest = {
        'analysis_name': CHARACTERIZATION_NAME,
        'campaign_id': CAMPAIGN_ID,
        'source_dataset': DATASET_NAME,
        'fit_analysis_root': str(SCALING_OUTPUT_ROOT / CAMPAIGN_ID),
        'targets': {
            'entropy_slope': ENTROPY_TARGET,
            'correlator_exponent_abs_slope': CORRELATOR_TARGET,
            'scatter_min_cycle': SCATTER_MIN_CYCLE,
            'rolling_window': ROLLING_WINDOW,
            'rolling_loglog_metrics': ['entropy_slope_rms_dev', 'correlator_exponent_rms_dev'],
            'histogram_start_cycle_exclusive': HISTOGRAM_START_CYCLE_EXCLUSIVE,
            'entropy_histogram_domain': [float(ENTROPY_HISTOGRAM_BINS[0]), float(ENTROPY_HISTOGRAM_BINS[-1])],
            'final_entropy_deviation_histogram_domain': [-0.1, 0.1],
            'spatial_charge_heatmap_frames_per_cycle': FRAME_REPEAT,
        },
        'records': [
            {
                'config_id': r['config_id'],
                'protocol': r['protocol'],
                'Nx': r['Nx'],
                'Ny': r['Ny'],
                'samples_actual': r['samples_actual'],
                'cycles': r['cycles'],
                'local_charge_cell_path': str(r['paths']['local_charge_cell']),
                'fit_npz_path': str(r['paths']['fit_npz']),
            }
            for r in payload['runs'].values()
        ],
        'skipped_runs': skipped_runs,
        'artifacts': _path_to_string(artifact_paths),
        'tables': _path_to_string(table_paths),
    }
    manifest_path = analysis_root / 'analysis_manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')
    print(f'[manifest] {manifest_path}')
else:
    manifest_path = None

display(Markdown('### Artifact Groups'))
display(pd.DataFrame([
    {'group': group, 'keys': list(paths.keys()) if isinstance(paths, dict) else paths}
    for group, paths in artifact_paths.items()
]))


In [ ]:
from google.colab import runtime
runtime.unassign()